# Silver to Gold: Enrichment and Dimension Table Processing

This notebook performs the gold-layer processing for all dimension tables in the e-commerce medallion architecture. It reads cleaned silver tables, enriches them with additional attributes and joins, and writes the final enriched dimension tables to the gold schema.

## Gold Dimension Tables

| Dimension | Silver Table | Gold Table | Rows | Enrichment Applied |
| --- | --- | --- | --- | --- |
| Products | `silver.slv_products` | `gold.gld_dim_products` | 50,000 | Join with brands (brand_name) and category (category_name) tables |
| Customers | `silver.slv_customers` | `gold.gld_dim_customers` | 299,700 | Add `region` column via state-to-region mapping for 7 countries |
| Date | `silver.slv_date` | `gold.gld_dim_date` | 92 | Add `date_id` (integer), `is_weekend` flag, and `month_name` columns |

All tables are in the `ecommerce` catalog.

## Standard Processing Pattern

Each dimension follows the same pipeline:

1. **Read** from the silver Delta table
2. **Enrich** — join with other tables or add derived columns to add analytical value
3. **Validate** — check for nulls, verify schema, and preview the enriched data
4. **Reorder** — arrange columns in a logical order for downstream consumption
5. **Write to Gold** — save as a managed Delta table with `mode("overwrite")`

In [0]:
# Import required libraries for gold-layer processing and transformations

import pyspark.sql.functions as F
import json
from pyspark.sql.types import (
    StringType,
    IntegerType,
    DateType,
    TimestampType,
    FloatType,
)
from pyspark.sql import Row

In [0]:
# Define the catalog name for the e-commerce project
catalog_name = 'ecommerce'

### Products Dim — Enrich with Brand and Category Names

Enriches the products dimension by joining the silver `slv_products` table with `slv_brands` (on `brand_code`) and `slv_category` (on `category_code`) to add `brand_name` and `category_name` columns. The result is written to `gold.gld_dim_products`.

> **Note:** The PySpark write cell is currently commented out. An equivalent SQL approach is also provided (also commented out) as an alternative.

The `category` table has `category_code` and `category_name`, and the `brand` table has `brand_code`, `brand_name`, and `category_code`.
The `products` table also has a `category_code` column, which allows us to join all three tables together.

**Steps:**
1. Join the `products` table with the `brands` table on `brand_code` to get the brand name.
2. Join the result with the `category` table on `category_code` to get the category name.
3. Select the relevant columns and save the enriched table to the gold layer.

In [0]:
# Read the brands, category, and products tables from the silver layer
df_brands = spark.read.table(f"{catalog_name}.silver.slv_brands")
df_brands.show(5)

df_category = spark.read.table(f"{catalog_name}.silver.slv_category")
df_category.show(5)

df_products = spark.read.table(f"{catalog_name}.silver.slv_products")
df_products.show(5)

To make the enriched gold-layer products table, we will join the brands table with the products table on `brand_code`, and then join the category table on `category_code`. This will give us a products dimension table with both brand and category names.

In [0]:
# Join products with brands and category tables to enrich the products dimension
df_gold = (
    df_products.alias('p')
    .join(
        df_brands.alias('b'),
        'brand_code',
        'left'
    )
    .join(
        df_category.alias('c'),
        'category_code',
        'left'
    )
    .select(
        'p.product_id',
        'p.sku',
        'p.category_code',
        'c.category_name',
        'p.brand_code',
        'b.brand_name',
        'p.color',
        'p.size',
        'p.material',
        'p.weight_grams',
        'p.length_cms',
        'p.width_cms',
        'p.height_cms',
        'p.rating_count',
        'p._source_file',
        'p._ingested_at'
    )
)

df_gold.show()

In [0]:
# Write the enriched products table to the gold layer

# df_gold.write \
#         .mode('overwrite') \
#         .format('delta') \
#         .option('overwriteSchema', True) \
#         .saveAsTable(f"{catalog_name}.gold.gld_dim_products")

**SQL Alternative:**

The same enrichment can also be done using SQL. To do that we need to register the silver DataFrames as temporary views, set the catalog, and then use a `CREATE OR REPLACE TABLE` statement with joins.

In [0]:
# Register the silver DataFrames as temporary views for SQL-based joins

# df_brands.createOrReplaceTempView("v_brands")

# df_category.createOrReplaceTempView("v_category")

# df_products.createOrReplaceTempView("v_products")

In [0]:
# Set the catalog before using SQL to create the gold-layer table

# spark.sql(f"USE CATALOG {catalog_name}")

In [0]:
%sql
-- Build brand-category mapping and join it with the products table, then save to the gold schema.
-- This will also create the table in the gold layer.

-- CREATE OR REPLACE TABLE gold.gld_dim_products AS

-- SELECT
--     p.product_id,
--     p.sku,
--     p.category_code,
--     c.category_name,
--     p.brand_code,
--     b.brand_name,
--     p.color,
--     p.size,
--     p.material,
--     p.weight_grams,
--     p.length_cms,
--     p.width_cms,
--     p.height_cms,
--     p.rating_count,
--     p._source_file,
--     p._ingested_at

-- FROM v_products AS p

-- LEFT JOIN v_brands AS b
--     ON p.brand_code = b.brand_code

-- LEFT JOIN v_category AS c
--     ON p.category_code = c.category_code;

### Customers Dim — Enrich with Regional Information

Enriches the customers dimension by adding a `region` column derived from a state-to-region mapping covering 7 countries (India, Australia, UK, US, UAE, Singapore, Canada). States not found in the mapping are assigned 'Other'. The result is written to `gold.gld_dim_customers`.

In [0]:
# Read the customers dimension table from the silver layer
df_customers = spark.read.table(f"{catalog_name}.silver.slv_customers")

df_customers.show(5)
print(f"Row count: {df_customers.count()}")
print(f"Column count: {len(df_customers.columns)}")

We want to add regional details to the customer table. For that we need a state-to-region mapping for all the countries, and then we can use that mapping to create a new `region` column in the customers table.

In [0]:
# Define state-to-region mappings for each country in the dataset

# India states
india_region = {
    "MH": "West",
    "GJ": "West",
    "RJ": "West",
    "KA": "South",
    "TN": "South",
    "TS": "South",
    "AP": "South",
    "KL": "South",
    "UP": "North",
    "WB": "North",
    "DL": "North",
}
# Australia states
australia_region = {"VIC": "SouthEast", "WA": "West", "NSW": "East", "QLD": "NorthEast"}

# United Kingdom states
uk_region = {
    "ENG": "England",
    "WLS": "Wales",
    "NIR": "Northern Ireland",
    "SCT": "Scotland",
}

# United States states
us_region = {
    "MA": "NorthEast",
    "FL": "South",
    "NJ": "NorthEast",
    "CA": "West",
    "NY": "NorthEast",
    "TX": "South",
}

# UAE states
uae_region = {"AUH": "Abu Dhabi", "DU": "Dubai", "SHJ": "Sharjah"}

# Singapore states
singapore_region = {"SG": "Singapore"}

# Canada states
canada_region = {
    "BC": "West",
    "AB": "West",
    "ON": "East",
    "QC": "East",
    "NS": "East",
    "IL": "Other",
}

In [0]:
# Create a combined country-state-region mapping dictionary
country_state_region_map = {
    "India": india_region,
    "Australia": australia_region,
    "UK": uk_region,
    "US": us_region,
    "UAE": uae_region,
    "Singapore": singapore_region,
    "Canada": canada_region,
}

print(json.dumps(country_state_region_map, indent=4))

In [0]:
# Convert the mapping dictionary into a Spark DataFrame for joining with the customers table
rows = []
for country, states in country_state_region_map.items():
    for state_code, region in states.items():
        rows.append(Row(country=country, state=state_code, region=region))

df_region_map = spark.createDataFrame(rows)

df_region_map.show(5)

In [0]:
# Join the region mapping DataFrame with the customers table on country and state
df_gold = df_customers.join(df_region_map, on=['country', 'state'], how='left')

df_gold.filter(F.col('region').isNull()).count()

In [0]:
# Some region values are null due to unmapped state codes — fill them with 'Other'
df_gold = df_gold.fillna({'region': 'Other'})

df_gold.filter(F.col('region').isNull()).count()

In [0]:
# Verify the final schema and preview the data before saving
df_gold.show(10, truncate=False)
df_gold.printSchema()

In [0]:
# Reorder the columns before saving to the gold layer
df_gold = df_gold.select(
    'customer_id',
    'phone',
    'country_code',
    'country',
    'state',
    'region',
    '_source_file',
    '_ingested_at'
)

df_gold.show(10, truncate=False)

In [0]:
# Write the enriched customers table to the gold layer
df_gold.write \
        .mode('overwrite') \
        .format('delta') \
        .option('mergeSchema', True) \
        .saveAsTable(f"{catalog_name}.gold.gld_dim_customers")

### Date Dim — Add Analytical Columns

Enhances the date dimension by adding three derived columns:
* `date_id` — integer representation of the date (e.g., `20250806`) for use as a join key
* `is_weekend` — binary flag (1 for Saturday/Sunday, 0 otherwise)
* `month_name` — full month name (e.g., `August`) extracted from the date

The result is written to `gold.gld_dim_date`.

We will enhance the date table with analytics columns such as `is_weekend`, `date_id`, and `month_name`.

**Steps:**
1. Add a `date_id` column by removing hyphens from the date and casting to integer.
2. Add an `is_weekend` flag by checking if the day name is Saturday or Sunday.
3. Add a `month_name` column using the full month name from the date.

In [0]:
# Read the date dimension table from the silver layer
df_date = spark.read.table(f"{catalog_name}.silver.slv_date")

df_date.show(10)
df_date.printSchema()

In [0]:
# Add date_id, is_weekend, and month_name columns to the date table
df_date = df_date.withColumn(
    "date_id",
    F.regexp_replace(F.col("date"), "-", "").cast(IntegerType())
)

# adding is_weekend column
df_date = df_date.withColumn(
    "is_weekend",
    F.when(F.col('day_name').isin("Saturday", "Sunday"), 1).otherwise(0)
)

# adding month_name column
df_date = df_date.withColumn(
    "month_name",
    F.date_format(F.col("date"), "MMMM").cast(StringType())
)

df_date.show(20, truncate=False)

In [0]:
# Reorder the columns before saving the table to the gold layer
df_date = df_date.select(
    'date_id',
    'date',
    'year',
    'month_name',
    'day_name',
    'is_weekend',
    'quarter',
    'quarter_number',
    'week_of_year',
    'week_number',
    '_source_file',
    '_ingested_at'
)

df_date.show(10, truncate=False)

In [0]:
# Write the enriched date table to the gold layer
df_date.write.mode("overwrite") \
            .format("delta") \
            .option('mergeSchema', True) \
            .saveAsTable(f"{catalog_name}.gold.gld_dim_date")

In [0]:
%sql
-- Verify the schema and metadata of the gold date table
DESCRIBE EXTENDED ecommerce.gold.gld_dim_date

## Summary

All three dimension tables have been enriched and written to the Gold layer:

| Gold Table | Source | Rows | Columns Added | Key Enrichment |
| --- | --- | --- | --- | --- |
| `ecommerce.gold.gld_dim_products` | `slv_products` | 50,000 | `brand_name`, `category_name` | Joined with brands and category tables on `brand_code` and `category_code` |
| `ecommerce.gold.gld_dim_customers` | `slv_customers` | 299,700 | `region` | State-to-region mapping for 7 countries; unmapped states filled with 'Other' |
| `ecommerce.gold.gld_dim_date` | `slv_date` | 92 | `date_id`, `is_weekend`, `month_name` | Derived analytical columns from the date value |

### Notes

* The **products** write cell is currently commented out — the enrichment DataFrame `df_gold` is built but not yet persisted to `gold.gld_dim_products`. An equivalent SQL alternative is also provided (commented out).
* The **customers** enrichment had 87,150 rows with null regions after the initial join (due to country name mismatches between the mapping and the source data, e.g., 'UK' vs 'United Kingdom'). These were filled with 'Other'.
* The **date** `date_id` column (e.g., `20250806`) serves as the integer join key used by the fact table in downstream Gold views.